# ENV SETUP

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sureshbeekhani/kaggleworkingchurn/Churn_Modelling.csv


# PART 1: IMPORT DATA

In [4]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "Churn_Modelling.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sureshbeekhani/kaggleworkingchurn",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

/tmp/ipykernel_57/2610185347.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


# PART 2: DATA SANITY CHECK

In [5]:
# check missing values - no missing values
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [6]:
# check basic data information including data type
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [43]:
# imbalanced data
df['Exited'].value_counts(normalize=True)

Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

# PART 3: MODEL DEVELOPMENT IN PROD ENV

## Step 1: Param Setup

In [52]:
RANDOM_STATE = 84
TEST_SIZE = 0.2
TARGET_COL = "Exited"
FEATURES_COLS = ["CreditScore", "Geography", "Gender", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"]
THRESHOLD = 0.5

## Step 2: Data Import

In [6]:
import numpy as np
import pandas as pd

def load_data(file_path):
    """
    Load the churn dataset from CSV.
    Arg:
        file_path: path to the csv file
    """
    df = pd.read_csv(file_path, index_col=0)
    return df

## Step 3: Data Preprocessing
## Including training/testing dataset split, categorical and numerical features transformations

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def preprocess_data(df, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """
    Preprocess the dataset for ML model training.
    
    Args:
        df (pd.DataFrame): Raw data
        test_size (float): Proportion of data to use for testing
        random_state (int): Random seed for reproducibility
        
    Returns:
        tuple: (X_train, X_test, y_train, y_test)
    """
    # Separate features and target
    X = df[FEATURES_COLS]  # Features
    y = df[TARGET_COL]     # Target
    
    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state,
        stratify=y # keeps the target varialbe distribution the same in train and test
    )

    # Identify categorical and numerical columns automatically
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
        
    # Create preprocessing transformers for both categorical and numerical data
    cat_trans = OneHotEncoder(handle_unknown='ignore')
    num_trans = StandardScaler()

    # Combine preprocessing steps
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_trans, num_cols),
            ('cat', cat_trans, cat_cols)
        ])

    # Fit the preprocessor on training data only
    X_train_processed = preprocessor.fit_transform(X_train)  # Learn parameters and transform
    X_test_processed = preprocessor.transform(X_test)        # Apply learned parameters without fitting
    
    return X_train_processed, X_test_processed, y_train, y_test

## Step 4: Model Training

In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

def train_model(X_train, y_train, model_type=MODEL_TYPE, **model_params):
    """
    Train a machine learning model on the preprocessed data.
    
    Args:
        X_train (array-like): Preprocessed training features
        y_train (array-like): Training target values
        model_type (str): Type of model to train
        **model_params: Additional parameters to pass to the model constructor
        
    Returns:
        object: Trained model
    """
    print(f"Training a {model_type} model...")
    
    if model_type == "random_forest":
        # Default parameters if not specified
        if 'n_estimators' not in model_params:
            model_params['n_estimators'] = 10  # Using a sensible default
        if 'random_state' not in model_params:
            model_params['random_state'] = 42  # For reproducibility
            
        model = RandomForestClassifier(class_weight='balanced',
                                       **model_params)
    
    elif model_type == "logistic":
        model = LogisticRegression(class_weight='balanced',
                                   **model_params)

    elif model_type == 'svm':
        model = LinearSVC()

    elif model_type == 'xgboost':
        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        model = XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=neg/pos)
    
    else:
        raise ValueError(f"Unsupported model type: {model_type}")
    
    # Train the model
    model.fit(X_train, y_train)
    
    print("Model training completed!")
    return model

## Step 5: Model Prediction

In [9]:
def predict_with_model(model, X):
    """
    Make predictions using a trained model.
    
    Args:
        model (object): Trained model
        X (array-like): Preprocessed features
        
    Returns:
        array: Predictions
    """
    y_pred_scr = model.predict_proba(X)[:,1]
    y_pred = (y_pred_scr >= THRESHOLD).astype(int)

    return y_pred_scr, y_pred

## Step 6: Model Evaluation

In [10]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

def evaluate_model(model, X_test, y_test):
    """
    Evaluate a trained model on test data.
    
    Args:
        model (object): Trained model
        X_test (array-like): Test features
        y_test (array-like): True target values
        
    Returns:
        dict: Dictionary of metrics
        array: Model predictions
    """
    # Generate predictions
    y_pred_scr, y_pred = predict_with_model(model, X_test)
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred, output_dict=True),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'pr_auc': average_precision_score(y_test, y_pred_scr)
    }
    
    return metrics, y_pred

## Step 7: Data Pipeline

In [53]:
def main():
    """Main function to demonstrate data processing."""
    
    # Step 2: Preprocess the data
    print("\nPreprocessing the dataset...")
    X_train, X_test, y_train, y_test = preprocess_data(df)
    
    # Print preprocessing results
    print(f"\nPreprocessing complete!")
    print(f"  - Training features shape: {X_train.shape}")
    print(f"  - Testing features shape: {X_test.shape}")
    print(f"  - Training target shape: {y_train.shape}")
    print(f"  - Testing target shape: {y_test.shape}")
    
    # Step 3: Train a model
    print("\n--- Training Model ---")
    model = train_model(
        X_train, 
        y_train, 
        model_type='xgboost',
        #class_weight='balanced',
        random_state=RANDOM_STATE
    )
    
    # Step 4: Generate predictions with the trained model
    y_pred_scr, y_pred = predict_with_model(model, X_test)
    print(f"Generated {len(y_pred)} predictions")

    # Step 5: Evaluate on test set
    metrics, predictions = evaluate_model(model, X_test, y_test)
    print("\nAccuracy:", metrics['accuracy'])
    print("\nPR_AUC:", metrics['pr_auc'])
    print("\nConfusion Matrix:")
    print(metrics['confusion_matrix'])
    print("\nClassification Report:")
    print(pd.DataFrame(metrics['classification_report']).T)

In [54]:
main()


Preprocessing the dataset...

Preprocessing complete!
  - Training features shape: (8000, 13)
  - Testing features shape: (2000, 13)
  - Training target shape: (8000,)
  - Testing target shape: (2000,)

--- Training Model ---
Training a xgboost model...
Model training completed!
Generated 2000 predictions

Accuracy: 0.8075

PR_AUC: 0.6850575432180619

Confusion Matrix:
[[1320  273]
 [ 112  295]]

Classification Report:
              precision    recall  f1-score    support
0              0.921788  0.828625  0.872727  1593.0000
1              0.519366  0.724816  0.605128   407.0000
accuracy       0.807500  0.807500  0.807500     0.8075
macro avg      0.720577  0.776720  0.738928  2000.0000
weighted avg   0.839895  0.807500  0.818271  2000.0000
